In [1]:
import pandas as pd

In [2]:
def run_etl_pipeline(account_profiles_df, fraud_patterns_df, network_edges_df, time_series_stats_df, transactions_df):

    # Make copies to avoid modifying original loaded dataframes if needed elsewhere
    account_profiles_df = account_profiles_df.copy()
    fraud_patterns_df = fraud_patterns_df.copy()
    network_edges_df = network_edges_df.copy()
    time_series_stats_df = time_series_stats_df.copy()
    transactions_df = transactions_df.copy()

    # --- 1. Datetime Conversion ---
    time_series_stats_df['hour'] = pd.to_datetime(time_series_stats_df['hour'])
    transactions_df['timestamp'] = pd.to_datetime(transactions_df['timestamp'])

    # --- 2. Missing Value Handling ---
    # Using .loc for explicit copy modification to avoid FutureWarning
    network_edges_df.loc[:, 'ring_id'] = network_edges_df['ring_id'].fillna('Unknown')
    transactions_df.loc[:, 'fraud_pattern'] = transactions_df['fraud_pattern'].fillna('No Fraud Pattern')

    # --- 3. Date Feature Extraction ---
    transactions_df.loc[:, 'transaction_year'] = transactions_df['timestamp'].dt.year
    transactions_df.loc[:, 'transaction_month'] = transactions_df['timestamp'].dt.month
    transactions_df.loc[:, 'transaction_month_name'] = transactions_df['timestamp'].dt.month_name()
    transactions_df.loc[:, 'transaction_day_name'] = transactions_df['timestamp'].dt.day_name()

    time_series_stats_df.loc[:, 'stats_year'] = time_series_stats_df['hour'].dt.year
    time_series_stats_df.loc[:, 'stats_month'] = time_series_stats_df['hour'].dt.month
    time_series_stats_df.loc[:, 'stats_month_name'] = time_series_stats_df['hour'].dt.month_name()
    time_series_stats_df.loc[:, 'stats_day_name'] = time_series_stats_df['hour'].dt.day_name()

    # --- 4. Boolean Conversion ---
    bool_cols_account_profiles = ['is_high_risk', 'has_2fa', 'is_fraudster']
    for col in bool_cols_account_profiles:
        account_profiles_df[col] = account_profiles_df[col].astype(bool)

    network_edges_df['both_fraud'] = (
        network_edges_df['both_fraud'].astype(bool)
    )

    bool_cols_transactions = ['card_present', 'device_known', 'is_foreign_txn', 'has_2fa', 'is_fraud']
    for col in bool_cols_transactions:
        transactions_df[col] = transactions_df[col].astype(bool)

    time_series_stats_df['is_weekend'] = (
        time_series_stats_df['is_weekend'].astype(bool)
    )

    return account_profiles_df, fraud_patterns_df, network_edges_df, time_series_stats_df, transactions_df

# Extracting Datasets

In [3]:
# Define paths to raw CSV files
raw_data_path = '../data/raw/'


def load_dataset(path):
    df = pd.read_csv(f'{raw_data_path}{path}')
    return df

# Load all raw CSV files
account_profiles_df = load_dataset("account_profiles.csv")
fraud_patterns_df = load_dataset("fraud_patterns.csv")
network_edges_df = load_dataset("network_edges.csv")
time_series_stats_df = load_dataset("time_series_stats.csv")
transactions_df = load_dataset("transactions.csv")

print("All raw CSV files loaded.")

All raw CSV files loaded.


# Execute ETL Pipeline

In [4]:
# Call the ETL pipeline function with the raw DataFrames
account_profiles_df_processed, fraud_patterns_df_processed, network_edges_df_processed, time_series_stats_df_processed, transactions_df_processed = run_etl_pipeline(
    account_profiles_df, fraud_patterns_df, network_edges_df, time_series_stats_df, transactions_df
)

print("ETL pipeline executed. Processed DataFrames are available.")

ETL pipeline executed. Processed DataFrames are available.


# Verify Transformed Outputs

In [5]:
print('--- Account Profiles DataFrame (Processed) ---')
account_profiles_df_processed.info()
display(account_profiles_df_processed.head())

print('\n--- Network Edges DataFrame (Processed) ---')
network_edges_df_processed.info()
display(network_edges_df_processed.head())

print('\n--- Time Series Stats DataFrame (Processed) ---')
time_series_stats_df_processed.info()
display(time_series_stats_df_processed.head())

print('\n--- Transactions DataFrame (Processed) ---')
transactions_df_processed.info()
display(transactions_df_processed.head())

--- Account Profiles DataFrame (Processed) ---
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   account_id          50000 non-null  str    
 1   account_age_days    50000 non-null  int64  
 2   credit_limit        50000 non-null  float64
 3   home_country        50000 non-null  str    
 4   risk_score          50000 non-null  float64
 5   is_high_risk        50000 non-null  bool   
 6   avg_txn_amount      50000 non-null  float64
 7   avg_monthly_txns    50000 non-null  float64
 8   has_2fa             50000 non-null  bool   
 9   account_type        50000 non-null  str    
 10  total_transactions  50000 non-null  float64
 11  total_amount        50000 non-null  float64
 12  avg_amount          50000 non-null  float64
 13  max_amount          50000 non-null  float64
 14  fraud_count         50000 non-null  float64
 15  fraud_amount     

,account_id,account_age_days,credit_limit,home_country,risk_score,is_high_risk,avg_txn_amount,avg_monthly_txns,has_2fa,account_type,...,max_amount,fraud_count,fraud_amount,pct_foreign,avg_velocity,unique_countries,unique_categories,avg_ip_risk,fraud_rate,is_fraudster
0,ACC0000001,353,2171.42,US,16.5,False,90.82,71.1,True,personal,...,740.46,0.0,0.00,0.35,1.27,9.0,11.0,22.98,0.0,False
1,ACC0000002,2831,3031.38,US,25.4,False,63.78,7.4,True,business,...,186.13,1.0,186.13,0.50,4.00,2.0,2.0,52.45,0.5,True
2,ACC0000003,2399,7533.75,US,10.6,False,72.18,31.9,False,personal,...,1175.86,0.0,0.00,0.17,1.44,3.0,8.0,15.50,0.0,False
3,ACC0000004,1618,4821.94,US,20.7,False,53.64,16.6,True,personal,...,2753.31,0.0,0.00,0.33,1.11,4.0,10.0,19.01,0.0,False
4,ACC0000005,1597,3355.10,US,30.6,False,168.44,32.0,True,personal,...,718.61,0.0,0.00,0.42,0.95,7.0,12.0,26.71,0.0,False



--- Network Edges DataFrame (Processed) ---
<class 'pandas.DataFrame'>
RangeIndex: 7411 entries, 0 to 7410
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   account_a         7411 non-null   str  
 1   account_b         7411 non-null   str  
 2   shared_type       7411 non-null   str  
 3   connection_count  7411 non-null   int64
 4   ring_id           7411 non-null   str  
 5   both_fraud        7411 non-null   bool 
dtypes: bool(1), int64(1), str(4)
memory usage: 296.9 KB


,account_a,account_b,shared_type,connection_count,ring_id,both_fraud
0,ACC0017803,ACC0040032,phone,7,RING0001,True
1,ACC0017803,ACC0042246,email_domain,12,RING0001,True
2,ACC0017803,ACC0029491,phone,13,RING0001,True
3,ACC0017803,ACC0022213,phone,12,RING0001,True
4,ACC0017803,ACC0007601,ip_address,14,RING0001,True



--- Time Series Stats DataFrame (Processed) ---
<class 'pandas.DataFrame'>
RangeIndex: 26280 entries, 0 to 26279
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   hour               26280 non-null  datetime64[us]
 1   transaction_count  26280 non-null  int64         
 2   fraud_count        26280 non-null  int64         
 3   total_amount       26280 non-null  float64       
 4   avg_amount         26280 non-null  float64       
 5   avg_ip_risk        26280 non-null  float64       
 6   fraud_rate         26280 non-null  float64       
 7   hour_of_day        26280 non-null  int64         
 8   day_of_week        26280 non-null  int64         
 9   is_weekend         26280 non-null  bool          
 10  stats_year         26280 non-null  int32         
 11  stats_month        26280 non-null  int32         
 12  stats_month_name   26280 non-null  str           
 13  stats_day_name     2628

,hour,transaction_count,fraud_count,total_amount,avg_amount,avg_ip_risk,fraud_rate,hour_of_day,day_of_week,is_weekend,stats_year,stats_month,stats_month_name,stats_day_name
0,2022-01-01 00:00:00,40,1,9341.66,233.54,19.92,0.025000,0,5,True,2022,1,January,Saturday
1,2022-01-01 01:00:00,43,0,6295.77,146.41,20.32,0.000000,1,5,True,2022,1,January,Saturday
2,2022-01-01 02:00:00,41,1,10763.95,262.54,22.13,0.024390,2,5,True,2022,1,January,Saturday
3,2022-01-01 03:00:00,34,0,9354.50,275.13,22.00,0.000000,3,5,True,2022,1,January,Saturday
4,2022-01-01 04:00:00,39,2,5897.09,151.21,25.87,0.051282,4,5,True,2022,1,January,Saturday



--- Transactions DataFrame (Processed) ---
<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 27 columns):
 #   Column                  Non-Null Count    Dtype         
---  ------                  --------------    -----         
 0   transaction_id          1000000 non-null  str           
 1   account_id              1000000 non-null  str           
 2   timestamp               1000000 non-null  datetime64[us]
 3   hour_of_day             1000000 non-null  int64         
 4   day_of_week             1000000 non-null  int64         
 5   is_weekend              1000000 non-null  int64         
 6   amount                  1000000 non-null  float64       
 7   merchant_category       1000000 non-null  str           
 8   mcc_code                1000000 non-null  int64         
 9   merchant_country        1000000 non-null  str           
 10  card_present            1000000 non-null  bool          
 11  device_type             1000000 non-null  st

,transaction_id,account_id,timestamp,hour_of_day,day_of_week,is_weekend,amount,merchant_category,mcc_code,merchant_country,...,amount_vs_avg_ratio,account_age_days,has_2fa,credit_limit,is_fraud,fraud_pattern,transaction_year,transaction_month,transaction_month_name,transaction_day_name
0,TXN000000001,ACC0016173,2023-02-21 08:02:38,8,1,0,168.42,travel,4511,CA,...,2.6423,3256,True,3958.46,False,No Fraud Pattern,2023,2,February,Tuesday
1,TXN000000002,ACC0011196,2024-05-12 23:13:34,23,6,1,85.78,online_retail,5999,AU,...,0.7279,1527,True,3553.35,False,No Fraud Pattern,2024,5,May,Sunday
2,TXN000000003,ACC0001181,2023-09-22 23:28:21,23,4,0,20.15,pharmacy,5912,CA,...,0.1851,2230,True,4362.57,False,No Fraud Pattern,2023,9,September,Friday
3,TXN000000004,ACC0037105,2022-09-28 23:26:38,23,2,0,62.49,grocery,5411,US,...,1.5223,1863,True,3194.84,False,No Fraud Pattern,2022,9,September,Wednesday
4,TXN000000005,ACC0028471,2023-02-23 17:54:13,17,3,0,71.68,online_retail,5999,US,...,0.7724,1728,False,11850.06,False,No Fraud Pattern,2023,2,February,Thursday
